# Vaja 13: Numerično reševanje diferencialnih enačb - robni problem

In [ ]:
import numpy as np

### `scipy.integrate.solve_bvp`

```python
scipy.integrate.solve_bvp(fun, bc, x, y, 
                          p=None, S=None, fun_jac=None, bc_jac=None, 
                          tol=0.001, max_nodes=1000, verbose=0)
```

---

### Naloga 1: poševni met 

Izstrelek mase $m$ izstrelimo z neznano hitrostjo $v_0$ pod neznanim kotom $\alpha$. Gibalni enačbi za pomike v smereh $x$ in $y$ sta:

$$\ddot{x}(t) = - F~\frac{\cos(\alpha)}{m} \qquad \ddot{y}(t) = -F~\frac{\sin(\alpha)}{m} - g$$

Sila upora $F$ je enaka $F = c~v^{3/2}$, pri čemer je hitrost $v = \sqrt{\dot{x}^2 + \dot{y}^2}$ in $\alpha = \arctan{\Big(\frac{\dot{y}}{\dot{x}}\Big)}$

<img src="def_vaja12.png" width=300>
<center><font size="1.5">vir: Numerical Methods in Engineering With Python 3, 3rd Ed, Jaan Kiusalaas</font></center>

Izstrelek ste izstrelili **iz koordinatnega izhodišča** vzdolž vodoravnega polja, izmerili ste, da je **padel na tla po 6 sekundah** in pri tem **preletel razdaljo 250 m**.

Rešite sistem diferencialnih enačb in določite krivuljo (trajektorijo) leta izstrelka.

In [ ]:
# Podatki, naloga 1
g = 9.81 # m/s^2
c = 0.029 # N/(ms)^2
m = 0.41 # kg
T = 6 # s
D = 250 # m

#### Numerično reševanje: `scipy.integrate.solve_bvp`

In [ ]:
# Naloga 1, funkcija prvih odvodov - izstrelek
def fun(t, y):
    """
    Funkcija prvih odvodov za reševanje sistema D.E. poševnega meta.
    """
    alpha = np.arctan2(y[3], y[1])
    v = np.sqrt(y[3]**2 + y[1]**2)
    F = c*v**(3/2)
    return y[1], -F*np.cos(alpha)/m, y[3], -F*np.sin(alpha)/m-g

---

### Naloga 2: nosilec

<img src="konzola_s_podporo.png" width=450>

Diferencialna enačba (Euler Bernoulli):

$$EI \, y^{IV} = q(x)$$

**Robni pogoji:**

$$\begin{align}
y(A) = 0\\
y'(A) = 0\\
y(B) = 0\\
y''(B)  = 0
\end{align}$$

In [ ]:
# Podatki, naloga 2
E = 2.1e5 # N/mm2
I = 5.0e6 # mm^4
L = 1300  # mm
q = 150   # kg/mm

#### Simbolno

In [ ]:
# Naloga 2, simbolno reševanje
import sympy as sym
x, EI, q0 = sym.symbols('x EI q_0')
l = sym.symbols('l', positive=True)
y = sym.Function('y')(x)
eq = sym.Eq(EI*y.diff(x, 4), q0)
res = sym.dsolve(eq).args[1]
# Reševanje sistema robnih pogojev
en1 = sym.Eq(res.subs(x, 0), 0)
en2 = sym.Eq(sym.diff(res, x).subs(x, 0), 0)
en3 = sym.Eq(res.subs(x, l), 0)
en4 = sym.Eq(sym.diff(res, x, 2).subs(x, l), 0)
sistem = (en1, en2, en3, en4)
konstante = sym.solve(sistem, ['C1', 'C2', 'C3', 'C4'])
končna_rešitev = res.subs(konstante)
# Prikaz končne rešitve
podatki = {l:L, EI:E*I, q0:q}
poves = končna_rešitev.subs(podatki)
sym.plot(poves, (x, 0, L))
f_poves = sym.lambdify(x, poves)

#### Numerično - `scipy.integrate.solve_bvp`

---

### Naloga 3: Nosilec z dvema poljema

<img src="konzola_s_podporo_dve_polji.png" width=550>

**Polje I:**

Diferencialna enačba (Euler Bernoulli):

$$EI \, y_I^{IV} = q(x) = 0$$

Nove neznanke:

$$\begin{align}
y_0 = y_{I}(x)\\
y_1 = y_{I}'(x)\\
y_2 = y_{I}''(x)\\
y_3 = y_{I}'''(x)
\end{align}$$


**Polje II:**

Diferencialna enačba (Euler Bernoulli):

$$EI \, y_{II}^{IV} = q(x) = 0$$

Nove neznanke:

$$\begin{align}
y_4 = y_{II}(x)\\
y_5 = y_{II}'(x)\\
y_6 = y_{II}''(x)\\
y_7 = y_{II}'''(x)
\end{align}$$

**8 novih neznank!**

In [ ]:
# Podatki, naloga 3
L = 2000 # mm
E = 210000 # MPa
I = 2e6 # mm^4
F = -5000